# 02. 3D world point를 camera image에 투영하기

목표: HD map·trajectory polyline 생성의 핵심인 world-to-camera transform, pinhole projection, image clipping을 표준 Python으로 구현합니다. 실제 calibration 값이 아닌 이해용 toy parameter입니다.

In [ ]:
WIDTH, HEIGHT = 2048, 2464
fx, fy = 1000.0, 1000.0
cx, cy = WIDTH / 2, HEIGHT / 2
R = [
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
]
t = [0.0, 0.0, 0.0]
print({'image': (WIDTH, HEIGHT), 'focal': (fx, fy), 'principal_point': (cx, cy)})

In [ ]:
def matvec(matrix, vector):
    return [sum(row[index] * vector[index] for index in range(3)) for row in matrix]

def world_to_camera(point):
    rotated = matvec(R, point)
    return [rotated[index] + t[index] for index in range(3)]

def project(point):
    x, y, z = world_to_camera(point)
    if z <= 0:
        return None  # camera 뒤의 점은 보이지 않습니다.
    u = fx * x / z + cx
    v = fy * y / z + cy
    visible = 0 <= u < WIDTH and 0 <= v < HEIGHT
    return {'pixel': (u, v), 'depth': z, 'visible': visible}

points = [(-1.0, 0.0, 10.0), (1.0, 0.0, 10.0), (0.0, 2.0, 8.0), (8.0, 0.0, 5.0), (0.0, 0.0, -2.0)]
for point in points:
    print(point, '->', project(point))

In [ ]:
results = [project(point) for point in points]
assert results[0]['visible'] and results[1]['visible'] and results[2]['visible']
assert not results[3]['visible']
assert results[4] is None
assert results[0]['pixel'][0] < cx < results[1]['pixel'][0]
print('projection 기본 조건 검증 완료')

## 실제 calibration에 적용할 때

1. Lanelet2 local coordinate를 EPSG:6677로 재투영합니다.
2. COLMAP quaternion을 rotation matrix로 바꿉니다.
3. 원문의 convention처럼 `X_camera = R X_world + t`인지 확인합니다.
4. `z <= 0`인 점을 제거하고 image boundary에서 segment를 clip합니다.
5. 연속 visibility가 끊기면 polyline도 분리합니다.
6. 몇 frame을 수동 overlay해 좌우 반전·axis swap·unit 오류를 찾습니다.

In [ ]:
def visible_runs(polyline):
    runs, current = [], []
    for point in polyline:
        result = project(point)
        if result is not None and result['visible']:
            current.append(result['pixel'])
        elif current:
            runs.append(current); current = []
    if current:
        runs.append(current)
    return runs

toy_lane = [(-2, 0, 8), (-1, 0, 8), (0, 0, -1), (1, 0, 8), (2, 0, 8)]
runs = visible_runs(toy_lane)
print('visible polyline runs:', runs)
assert len(runs) == 2